# Part 1 — SFT Data Curation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoodm2/LLM-Lab/blob/main/notebooks/01_data_sft.ipynb)

Prepares the training dataset for QLoRA SFT in `02_sft.ipynb`.

**Source**: `ise-uiuc/Magicoder-OSS-Instruct-75K` — 75k high-quality code instruction pairs generated by GPT-4 from real open-source code snippets (OSS-Instruct method).  
**Output**: filtered + formatted dataset saved to HuggingFace Hub as `{HF_USERNAME}/magicoder-qwen-sft`.

**What this notebook does:**
1. Load raw Magicoder dataset
2. Filter: length, quality, dedup
3. Format: apply Qwen2.5 chat template, mask prompt tokens from loss
4. Inspect: verify token distribution and a few samples
5. Push to HF Hub

## 1. Setup

In [ ]:
!pip install -q transformers datasets huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
from collections import Counter
import matplotlib.pyplot as plt
import re

HF_USERNAME = "YOUR_HF_USERNAME"   # <-- replace
HF_DATASET_ID = f"{HF_USERNAME}/magicoder-qwen-sft"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_TOKENS = 1024   # discard examples longer than this
MIN_RESPONSE_CHARS = 50  # discard trivially short responses

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Vocab size: {tokenizer.vocab_size}")

## 2. Load Raw Dataset

In [ ]:
raw = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
print(f"Raw size: {len(raw):,}")
print(raw[0])

## 3. Filter

Three passes:
- **Length**: discard examples that tokenize beyond MAX_TOKENS (would require padding/truncation that wastes compute)
- **Quality**: discard responses shorter than MIN_RESPONSE_CHARS (likely malformed)
- **Dedup**: exact dedup on the problem field (some OSS snippets appear multiple times)

In [ ]:
def make_messages(example):
    """Convert raw Magicoder fields to chat messages."""
    return [
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": example["solution"]},
    ]


def token_length(example):
    messages = make_messages(example)
    ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
    return len(ids)


# Quality filter (fast, no tokenization)
def quality_filter(example):
    if len(example["solution"].strip()) < MIN_RESPONSE_CHARS:
        return False
    if not example["problem"].strip():
        return False
    return True


filtered = raw.filter(quality_filter)
print(f"After quality filter: {len(filtered):,} (removed {len(raw)-len(filtered):,})")

In [ ]:
# Length filter — tokenize each example (slower, batched for speed)
filtered = filtered.map(
    lambda ex: {"token_len": token_length(ex)},
    num_proc=2,
    desc="Tokenizing for length filter",
)
before = len(filtered)
filtered = filtered.filter(lambda ex: ex["token_len"] <= MAX_TOKENS)
print(f"After length filter (≤{MAX_TOKENS} tokens): {len(filtered):,} (removed {before-len(filtered):,})")

In [ ]:
# Exact dedup on problem field
seen = set()
def dedup(example):
    key = example["problem"].strip()
    if key in seen:
        return False
    seen.add(key)
    return True

before = len(filtered)
filtered = filtered.filter(dedup)
print(f"After dedup: {len(filtered):,} (removed {before-len(filtered):,} duplicates)")

## 4. Inspect Token Distribution

In [ ]:
lengths = filtered["token_len"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=50, color="#3498db", edgecolor="white")
ax.axvline(MAX_TOKENS, color="red", linestyle="--", label=f"cutoff ({MAX_TOKENS})")
ax.set_xlabel("Token length")
ax.set_ylabel("Count")
ax.set_title(f"Token length distribution — {len(filtered):,} examples")
ax.legend()
plt.tight_layout()
plt.show()

import statistics
print(f"Mean: {statistics.mean(lengths):.0f}")
print(f"Median: {statistics.median(lengths):.0f}")
print(f"P95: {sorted(lengths)[int(len(lengths)*0.95)]:.0f}")
print(f"Max: {max(lengths)}")

## 5. Apply Qwen Chat Template + Loss Masking

Key: `input_ids` contains the full conversation; `labels` is a copy with prompt tokens set to -100.  
The SFTTrainer computes cross-entropy loss only where `labels != -100`, so the model learns to generate responses, not regurgitate prompts.

`apply_chat_template` with `tokenize=True` returns the full sequence. To find where the assistant turn starts, we look for the assistant header token sequence in the input_ids.

In [ ]:
# Find the token ids that mark the start of the assistant response
# Qwen2.5 uses: <|im_start|>assistant\n
ASSISTANT_HEADER = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
print(f"Assistant header tokens: {ASSISTANT_HEADER}")
print(f"Decoded: {tokenizer.decode(ASSISTANT_HEADER)}")

In [ ]:
def find_assistant_start(input_ids: list, header: list) -> int:
    """Return index of first token after the last assistant header in input_ids."""
    n = len(header)
    for i in range(len(input_ids) - n, -1, -1):  # search from end (last turn)
        if input_ids[i:i+n] == header:
            return i + n
    return -1  # should never happen with well-formed data


def format_example(example):
    messages = make_messages(example)
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
    )
    labels = [-100] * len(input_ids)  # mask everything by default
    assistant_start = find_assistant_start(input_ids, ASSISTANT_HEADER)
    if assistant_start > 0:
        labels[assistant_start:] = input_ids[assistant_start:]  # unmask response
    return {"input_ids": input_ids, "labels": labels}


formatted = filtered.map(
    format_example,
    remove_columns=filtered.column_names,
    num_proc=2,
    desc="Applying chat template",
)
print(f"Formatted dataset: {len(formatted):,} examples")
print(f"Columns: {formatted.column_names}")

## 6. Sanity Check

Verify that:
1. `input_ids` and `labels` have the same length
2. Prompt tokens are masked (`labels == -100`)
3. Response tokens are unmasked and match `input_ids`

In [ ]:
sample = formatted[0]
ids = sample["input_ids"]
lbls = sample["labels"]

assert len(ids) == len(lbls), "Length mismatch!"

# Find where masking ends
first_unmasked = next(i for i, l in enumerate(lbls) if l != -100)
print(f"Total tokens: {len(ids)}")
print(f"Masked (prompt): {first_unmasked} tokens")
print(f"Unmasked (response): {len(ids) - first_unmasked} tokens")
print()
print("--- Prompt (last 20 tokens before response) ---")
print(tokenizer.decode(ids[max(0, first_unmasked-20):first_unmasked]))
print("--- Response start (first 50 tokens) ---")
print(tokenizer.decode(ids[first_unmasked:first_unmasked+50]))

In [ ]:
# Check a few examples for masking correctness
for i in range(3):
    ex = formatted[i]
    masked = sum(1 for l in ex["labels"] if l == -100)
    unmasked = sum(1 for l in ex["labels"] if l != -100)
    print(f"Example {i}: {masked} masked (prompt), {unmasked} unmasked (response), total {len(ex['input_ids'])}")

## 7. Push to HuggingFace Hub

In [ ]:
formatted.push_to_hub(HF_DATASET_ID, private=True)
print(f"Dataset pushed to: https://huggingface.co/datasets/{HF_DATASET_ID}")
print(f"Total examples: {len(formatted):,}")

## Summary

| Step | Count |
|------|-------|
| Raw Magicoder | 75,000 |
| After quality filter | see above |
| After length filter | see above |
| After dedup | see above |
| **Final training set** | **see above** |

Next: `02_sft.ipynb` — load this dataset and run QLoRA SFT with Unsloth + TRL `SFTTrainer`.